In [ ]:
pip install yt-dlp


In [ ]:
!yt-dlp -o "dj_set.%(ext)s" "https://www.youtube.com/watch?v=c0-hvjV2A5Y"

In [ ]:
!pip install clip

In [ ]:
!pip install PIL

In [ ]:
import glob

VIDEO_DIR   = "/Users/fokia/CentraleSupelec-CV-and-RL-project/dj_sets_videos"
OUTPUT_BASE = "/Users/fokia/CentraleSupelec-CV-and-RL-project/frames"

all_frames = []

for video_path in sorted(glob.glob(f"{VIDEO_DIR}/*.mp4")):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    output_dir = os.path.join(OUTPUT_BASE, video_name)

    frames_df = extract_frames(video_path, output_dir, fps=2)
    frames_df["video"] = video_name   # track which video each frame came from
    all_frames.append(frames_df)

all_frames_df = pd.concat(all_frames, ignore_index=True)
print(all_frames_df)


In [ ]:
def filter_frames(frames_dir, output_dir, threshold=0.04):
    os.makedirs(output_dir, exist_ok=True)
    kept, total = 0, 0

    for fname in sorted(os.listdir(frames_dir)):
        if not fname.endswith('.jpg'):
            continue
        path = os.path.join(frames_dir, fname)
        frame = cv2.imread(path)
        if frame is None:          # guard against unreadable files too
            continue
        total += 1

        if has_dj_equipment(frame, threshold):
            cv2.imwrite(os.path.join(output_dir, fname), frame)
            kept += 1

    if total == 0:
        print(f"No frames found in {frames_dir}")
        return
    print(f"Kept {kept}/{total} frames ({100*kept/total:.1f}%)")


In [ ]:
for video_path in sorted(glob.glob(f"{VIDEO_DIR}/*.mp4")):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    frames_dir   = os.path.join(OUTPUT_BASE, video_name)
    filtered_dir = os.path.join("filtered_frames", video_name)  # mirrors structure
    filter_frames(frames_dir, filtered_dir, threshold=0.04)
